## Libraries

In [1]:
import fast_text_splitter as fts
from langchain_text_splitters import RecursiveCharacterTextSplitter
import recursive_splitter_rust
import time
import random

## Splitter Definitions

In [2]:
# Cache basic setup
conf_params = fts.py_ws_config_params(None,["\n\n", "\n"],8, max_depth=10, parallel=False)
data = "Hello, you all! \n How are you foo and poo? \n\n I am fine. Nice to meet you all insecure!"

In [3]:
t = time.time()
for i in range(1000):
    conf_params = fts.py_ws_config_params(None,["\n\n", "\n"], 8, max_depth=3, parallel=False)
    splits = fts.text_split_ws(data, conf_params, use_cache=True)
total = time.time() - t
print("FastTextSplitter (cached):         ", round(total * 1000, 2), "microseconds")

FastTextSplitter (cached):          12.01 microseconds


In [4]:
t = time.time()
for i in range(1000):
    conf_params = fts.py_ws_config_params(None,["\n\n", "\n"], 8, max_depth=3, parallel=False) 
    splits = fts.text_split_ws(data, conf_params, use_cache=False)
total = time.time() - t
print("FastTextSplitter (non-cached): ", round(total * 1000, 2), "microseconds")

FastTextSplitter (non-cached):  2.4 microseconds


In [5]:
if False:
    hf_conf_params = fts.py_hf_config_params(None,None,["\n\n", "\n"],32,2,True);
    splits = fts.text_split_hf(data, hf_conf_params)
    for split in splits:
        print("Split")
        print(split.split_strings)

# Comparison Superlinear

In [6]:
splitter_rust = recursive_splitter_rust.RecursiveTextSplitter(
    target_length=300,
    chunk_overlap=0,
    separators=["\n\n", "\n", ". ", " ", ""],
    is_regex=False,
    trim_trailing = True
)

In [7]:
conf_params = fts.py_ws_config_params(patterns=["\n\n", "\n", ". "], max_tokens=55, max_depth=3, merge_level=1, parallel=False) 

In [8]:
splitter_langchain = RecursiveCharacterTextSplitter(
    chunk_size=300,
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_overlap=0,
    keep_separator=True,
    is_separator_regex=False,
    strip_whitespace=True,
)

## Text Load

In [9]:
# Open file
with open("../../superlinear.txt", "r") as file:
    contents = file.read()

## Quality Compare

In [10]:
# Quality compare 
results_lc = splitter_langchain.create_documents([contents])
results_fast = fts.text_split_ws(contents, conf_params, use_cache=True)
results_rust = splitter_rust.split_texts([contents])[0]
print(len(results_lc), len(results_fast), len(results_rust), "\n")

for i, (text_lc, text_fast, text_rust) in enumerate(zip(results_lc, results_fast, results_rust)):
    print(f"Chunk {i + 1}:")
    print(f"Langchain: {text_lc.page_content}")
    print(f"FastText: {text_fast.split_strings}")
    print(f"RecursiveRust: {text_rust}")
    print("\n")
    if i == 2:
        break

123 111 116 

Chunk 1:
Langchain: October 2023

One of the most important things I didn't understand about the world when I was a child is the degree to which the returns for performance are superlinear.
FastText: October 2023

One of the most important things I didn't understand about the world when I was a child is the degree to which the returns for performance are superlinear.
RecursiveRust: October 2023

One of the most important things I didn't understand about the world when I was a child is the degree to which the returns for performance are superlinear.


Chunk 2:
Langchain: Teachers and coaches implicitly told us the returns were linear. "You get out," I heard a thousand times, "what you put in." They meant well, but this is rarely true. If your product is only half as good as your competitor's, you don't get half as many customers
FastText: 

Teachers and coaches implicitly told us the returns were linear. "You get out," I heard a thousand times, "what you put in." They mean

In [11]:
#for i, result in enumerate(results_fast):
#    print(f"Chunk {i + 1}: {result.split_strings}")

In [12]:
#for i, text in enumerate(results_rust):
#    print(f"Chunk {i + 1}: {text}")
#    print("-----------------------------")

## Speed Compare

In [13]:
copied = [contents]

In [14]:
%%timeit
results = splitter_langchain.create_documents(copied)

934 µs ± 6.46 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [15]:
%%timeit
results = splitter_rust.split_texts(copied)

46.9 µs ± 950 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [16]:
%%timeit
conf_params = fts.py_ws_config_params(patterns=["\n\n", "\n", ". "], max_tokens=55, max_depth=3, merge_level=1, parallel=False) 
results = fts.text_split_ws(copied[0], conf_params, use_cache=False)

130 µs ± 1.23 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [17]:
%%timeit
conf_params = fts.py_ws_config_params(patterns=["\n\n", "\n", ". "], max_tokens=55, max_depth=3, merge_level=1, parallel=False) 
results = fts.text_split_ws(copied[0], conf_params, use_cache=True)

119 µs ± 1.09 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [18]:
# 14.3 ms ± 348 µs per loop (mean ± std. dev. of 7 runs, 100 loops each) at 2000 randint and 100 loops
# 7.65 ms ± 330 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
